[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q3/00_orient_and_precommit.ipynb)

# R2-Q3 Week 1 — Orient and precommit your experimental design

### This notebook produces `precommit.json`, which holds the five decisions that drive every notebook after it.

This notebook walks you through R2-Q3's setup. You will read the question page and lock five decisions that fix the experiment before any classifier is trained: which failure-mode taxonomy you are mapping from, your experimental design, the exact augmentation list for the kitchen-sink baseline, the mapping from failure modes to targeted augmentations, and how you will compare results statistically. Locking these now means the head-to-head comparison in Week 4 stays honest — you cannot quietly swap in a different test or a different augmentation list after seeing the numbers.

By the end of this notebook you will have:

- A `precommit.json` file holding five decisions and the reasoning behind each: taxonomy source (R2-Q2 in parallel, R2-Q2's published version, or your own), experimental design (factorial vs hold-one-out), kitchen-sink composition (the exact augmentation list), the failure-mode-to-augmentation mapping (with the symptom-attended-but-wrong-class exclusion named explicitly per Consideration 2), and the statistical comparison (specific test, not just "compare").
- A short orientation note in your own words, restating R2-Q3 as you understand it.
- Submitted EQ#1.

In [ ]:
# Cell 0 — probe
try:
    import irilab2026 as iri
    print(f"irilab2026 {iri.__version__} already installed.")
    print("If you want to pull the latest changes, run Cell 1 below.")
except ModuleNotFoundError:
    print("irilab2026 not installed — run Cell 1 below.")

In [ ]:
# Cell 1 — install
# First run in a fresh Colab session: run this cell.
# Later runs in the same session: skip this cell.
!pip install git+https://github.com/geraldmc/irilab2026.git@main

**A note about a possible restart prompt.** If Colab shows a "RESTART SESSION" banner after the install: click it, wait for the kernel to reconnect, then continue from the next cell. The install does not need to be run again. If no banner appears, ignore this and move on.

In [ ]:
# Cell 2 — mount and setup
# Mount Drive, set up irilab2026, point to the R2-Q3 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r2_q3")
print(f"Output directory: {OUTPUT_DIR}")

### 1) Orient to the question


R2-Q3 is the third question in the Rationale 2 sequence, and it builds directly on the two before it. R2-Q1 measured the gap: a classifier trained on PlantVillage's lab images loses a large share of its accuracy when applied to PlantDoc's field photographs. R2-Q2 characterized *why* that gap exists, sorting the model's failures into a five-category taxonomy — background-attended, leaf-shape-attended, lighting-attended, symptom-attended-but-wrong-class, and other/ambiguous. R2-Q3 asks the next question: now that you know how the model fails, can you fix it?

The fix under test is image augmentation — transforming the training images so the model learns that small changes (a different background, a shift in lighting) shouldn't change the predicted disease. The question pits two augmentation strategies against each other. A **kitchen-sink** baseline applies a broad, general-purpose set of transformations without reference to any particular failure mode — the principle being that variety covers more ground than any single targeted fix. A **targeted** set is built deliberately from R2-Q2's failure-mode taxonomy: background-attended failures point toward background randomization, lighting-attended failures toward brightness and color jitter, and so on. You'll train three classifiers — no augmentation, kitchen-sink, and targeted — and compare how much each closes the PV→PD gap.

The prediction on the question page is worth holding in mind from the start: targeted is expected to win, but only by a modest margin — and a *small* margin is itself an informative result, because it suggests the kitchen-sink approach incidentally picked up the same signal the targeted set was designed to capture. The comparison is the headline finding, which is why Week 1 locks down the entire experimental design before any classifier is trained.

**Your turn.** Before moving to the precommits, write a short orientation note in your own words — three or four sentences restating what R2-Q3 is asking and why the three-way comparison is set up the way it is. This is the basis for your EQ#1 submission. Writing it now, before you lock any decisions, is a check on whether the question is clear to you: if you can't state it plainly, the precommit decisions in the sections below won't have a firm footing.

### 2) Precommit 1: taxonomy source (R2-Q2 in parallel, R2-Q2's published version, or your own)

This is the first of five decisions you lock in this notebook, and it comes first because it gates Precommit 4 — you can't write a failure-mode-to-augmentation mapping until you've decided which set of failure modes you're mapping from.

You have three options for where the taxonomy comes from:

- **R2-Q2 in parallel** — you (or a teammate) are working R2-Q2 in the same cohort, and you adopt the same five-category taxonomy it uses. This is the default below.
- **R2-Q2's published version** — you use the taxonomy as it appears in the R2-Q2 question page and reference materials, without running R2-Q2 yourself.
- **Your own** — you define a different set of failure modes, with your own justification.

In all three cases the mechanics are identical: you write the categories into *this* notebook's precommit. R2-Q3 does not read any file from R2-Q2 — the taxonomy is short enough that you restate it here, and the `taxonomy_source` field records which of the three options you chose so your methods section can report it honestly.

The five categories below are R2-Q2's. Four are named failure modes; the fifth, `other_ambiguous`, is the catch-all. If you're choosing "own," edit the list and set `taxonomy_source` to `"own"`. If you're choosing either R2-Q2 option, the list is already correct — change only the `taxonomy_source` string and the reasoning.

One category carries forward into Precommit 4 with a special status. `symptom_attended_but_wrong_class` is a failure where the model looked at a real symptom on the leaf but assigned the wrong disease — genuine confusion between similar-looking diseases, not a shortcut. Augmentation can't fix that, so when you build the mapping in Precommit 4 you'll name this category as deliberately unaddressed rather than quietly dropping it. Nothing to do about that here; just know the list you commit now has one member that's going to be set aside later, on purpose.

In [ ]:
# Initialize the running precommit dict.
# Each of the five sections below adds one top-level field; the closeout
# (Section 7) writes the whole thing to precommit.json in one json.dump.
precommit = {
    "taxonomy_source": None,         # Precommit 1 — this section
    "experimental_design": None,     # Precommit 2
    "kitchen_sink": None,            # Precommit 3
    "failure_mode_mapping": None,    # Precommit 4
    "statistical_comparison": None,  # Precommit 5
}

# Commit Precommit 1: the taxonomy source and the categories themselves.
# Default is "r2q2_parallel" with R2-Q2's five categories pre-filled.
# If you chose "own", edit `categories` and set `source` to "own".
# If you chose the published R2-Q2 taxonomy, set `source` to "r2q2_published".
precommit["taxonomy_source"] = {
    "source": "r2q2_parallel",   # one of: "r2q2_parallel", "r2q2_published", "own"
    "categories": [
        "background_attended",
        "leaf_shape_attended",
        "lighting_attended",
        "symptom_attended_but_wrong_class",
        "other_ambiguous",
    ],
    "reasoning": (
        # Replace this placeholder with your own reasoning before submission.
        # Cover at minimum: which of the three sources you chose and why; if
        # "own", what failure modes you added or removed relative to R2-Q2's
        # five and what evidence motivates each; and why these categories are
        # the right set to design augmentations against for the PV->PD gap.
        "PLACEHOLDER — REWRITE BEFORE SUBMISSION."
    ),
}

# Print back what just got committed.
ts = precommit["taxonomy_source"]
print(f"Taxonomy source: {ts['source']}")
print(f"Categories ({len(ts['categories'])}):")
for cat in ts["categories"]:
    print(f"  - {cat}")

### 3) Precommit 2: experimental design (factorial vs hold-one-out)

The three-way comparison — no augmentation, kitchen-sink, targeted — is the backbone of R2-Q3. But "targeted vs kitchen-sink" can be tested two different ways, and they answer slightly different questions. You lock which one here, before any training, because the design determines how many classifiers you train and what each comparison can claim.

- **Factorial** — you train the three named conditions as distinct, fully-specified pipelines: one with no augmentation, one with the complete kitchen-sink list, one with the complete targeted set. The comparison is clean and direct: three gap numbers, three conditions. What it answers: does the targeted set, as a whole, beat the kitchen-sink set, as a whole? This is the simpler design and the one the question page's Prediction is written against.
- **Hold-one-out** — you start from the kitchen-sink list and train additional classifiers that each drop one augmentation. Comparing each drop-one run against the full kitchen-sink tells you how much that single augmentation contributed. What it answers: which specific augmentations carry the gap reduction? This is more granular and more expensive — one extra training run per augmentation you want to probe.

The default below is factorial. It's the design that matches the headline question (targeted as a strategy vs kitchen-sink as a strategy), it keeps the training budget to three runs, and it's what NB01 through NB03 are scaffolded around. Choose hold-one-out only if your research question is specifically about *which* augmentations matter, and be aware it adds training runs NB01 isn't pre-built to loop over.

A note on what "factorial" means here, since the word has a stricter sense in statistics. A full factorial design would train every combination of every augmentation toggled on and off — far too many runs for a five-week timeline. What's meant here is the lighter sense: three pre-specified conditions compared head to head. If you commit to factorial, that's what you're committing to. Name it accurately in your reasoning so your methods section doesn't overclaim.

In [ ]:
# Commit Precommit 2: the experimental design.
# Default is "factorial" — the three named conditions compared head to head.
# Choose "hold_one_out" only if your question is about which specific
# augmentations carry the gap reduction (adds one training run per dropped
# augmentation; NB01 is not pre-built to loop over those extra runs).
precommit["experimental_design"] = {
    "design": "factorial",   # one of: "factorial", "hold_one_out"
    "n_conditions": 3,       # factorial: 3 (no-aug, kitchen-sink, targeted)
    "reasoning": (
        # Replace this placeholder with your own reasoning before submission.
        # Cover at minimum: which design you chose and what sub-question it
        # answers (targeted-vs-kitchen-sink as whole strategies, or which
        # individual augmentations matter); if "factorial", acknowledge you
        # mean the three-condition sense, not a full factorial over every
        # augmentation; and why that design fits your research question and
        # the five-week budget.
        "PLACEHOLDER — REWRITE BEFORE SUBMISSION."
    ),
}

# Print back what just got committed.
ed = precommit["experimental_design"]
print(f"Experimental design: {ed['design']}")
print(f"Number of conditions: {ed['n_conditions']}")

### 4) Precommit 3: kitchen-sink baseline composition (exact augmentation list, not "broad set")

The kitchen-sink baseline is the number the targeted set has to beat, so it has to be a specific procedure rather than a hand-wave at "a broad set of augmentations." If "kitchen-sink" stays vague, the Week 4 comparison has nothing solid on one side of it. You lock the exact list here.

Two reasonable ways to define it:

- **RandAugment** — an automated policy that samples from a fixed library of common transformations (rotation, shear, color adjustments, contrast, brightness, and others) with two parameters: how many transformations to apply per image, and how strong. It's well-validated, widely used, and a defensible default precisely because you didn't tune it to this task. The default below uses RandAugment.
- **A fixed explicit list** — a specific set of transformations drawn from a recent plant-disease paper or your own reading, each with its parameters written out. More transparent about exactly what's applied, at the cost of being a choice you have to defend transformation by transformation.

Either way, the principle from the question page holds: write the list down before you start, and don't change it once you've seen results. A kitchen-sink baseline that gets adjusted after you see the targeted numbers isn't a baseline anymore.

The point of the kitchen-sink condition isn't to be a strawman. It's a genuinely strong competitor — broad augmentation often reduces transfer gaps on its own, without any failure-mode reasoning behind it. That's exactly why the question page's Prediction says a *small* targeted-vs-kitchen-sink margin is informative rather than disappointing: it would mean kitchen-sink reached the same fix by brute variety. So commit a kitchen-sink list you'd genuinely expect to perform well, not a weak one chosen to make targeted look good.

In [ ]:
# Commit Precommit 3: the kitchen-sink baseline composition.
# Default is RandAugment with standard parameters — a well-validated,
# untuned broad-augmentation policy. The targeted set (Precommit 4) has to
# beat this on PV->PD gap reduction.
#
# If you prefer a fixed explicit list instead, set "method" to "fixed_list"
# and fill in "transforms" with named transformations and their parameters
# (e.g. {"name": "RandomRotation", "degrees": 30}). Leave "randaugment_params"
# as None in that case.
precommit["kitchen_sink"] = {
    "method": "randaugment",   # one of: "randaugment", "fixed_list"
    "randaugment_params": {
        "num_ops": 2,          # transformations applied per image
        "magnitude": 9,        # strength, on RandAugment's 0-30 scale
    },
    "transforms": None,        # used only when method == "fixed_list"
    "reasoning": (
        # Replace this placeholder with your own reasoning before submission.
        # Cover at minimum: which method you chose and why; if RandAugment,
        # why the num_ops and magnitude values you committed (and that you did
        # NOT tune them to this task); if a fixed list, why each transformation
        # belongs and where the list comes from; and why this is a strong
        # baseline rather than a weak one set up to lose.
        "PLACEHOLDER — REWRITE BEFORE SUBMISSION."
    ),
}

# Print back what just got committed.
ks = precommit["kitchen_sink"]
print(f"Kitchen-sink method: {ks['method']}")
if ks["method"] == "randaugment":
    p = ks["randaugment_params"]
    print(f"  RandAugment: num_ops={p['num_ops']}, magnitude={p['magnitude']}")
else:
    n = len(ks["transforms"]) if ks["transforms"] else 0
    print(f"  Fixed list: {n} transforms")

### 5) Precommit 4: failure-mode-to-augmentation mapping (written before any training; symptom-attended-but-wrong-class named as unaddressable per Consideration 2)

This is the heart of the targeted strategy. Precommit 1 fixed *which* failure modes you're working from; this section says what you'll *do* about each one. For every category in your taxonomy, you commit — in writing, before any training — which augmentation or augmentations are meant to counter it. NB02 reads this mapping and builds the targeted pipeline from it directly, so the mapping you write here *is* the targeted condition.

The logic of each mapping is "if the model fails by attending to X, train it on images where X varies, so X stops being a reliable cue." Worked through the four named categories:

- **background_attended** → background randomization or replacement. If the model keys on the background, train it on the same leaves against many different backgrounds so the background stops predicting the label.
- **leaf_shape_attended** → random cropping, shape distortion, or perspective changes. If the model keys on the leaf outline (a proxy for host species, which correlates with disease in PV), break the clean outline so shape stops being a shortcut.
- **lighting_attended** → brightness, contrast, and color jitter. If the model keys on PV's studio lighting, vary the lighting so the model can't rely on it.
- **symptom_attended_but_wrong_class** → nothing. This is the exclusion (next cell).

Two categories map to no augmentation, and they're in the dict for a reason — naming them as deliberately unaddressed is itself part of the commitment, per Consideration 2 on the question page. Dropping them silently would let the targeted design look more complete than it is.

- **symptom_attended_but_wrong_class** is unaddressable *because augmentation is the wrong tool*. The model looked at a real symptom and assigned the wrong disease — genuine confusion between similar-looking diseases. No amount of background or lighting variation fixes a decision boundary that's wrong about what the symptom means. Trying to "augment" this away would be a category error.
- **other_ambiguous** is unaddressable *because there's nothing specific to target*. By definition these are failures you couldn't sort into a named mode — there's no identified cue to vary. A large share landing here would be a finding about the taxonomy, not a target for augmentation.

Both carry `addressable: false` and an empty augmentation list. The difference between them lives in the `rationale` string — one is "wrong tool for a known problem," the other is "no known problem to aim at." Keeping both in the dict with the same structural shape means the targeted design accounts for all five categories on the record, even the two it deliberately does nothing about.

The augmentation names below are starting defaults, not the final word. The specific transforms — how strong the background randomization is, what crop scale, what jitter range — get implemented in NB02, where you can see them applied to real images. What you lock *here* is the mapping itself: which failure mode is countered by which kind of augmentation, and which are deliberately left alone. If you chose "own" for your taxonomy in Precommit 1, edit the keys below to match your categories, keeping the same shape (an `augmentations` list, an `addressable` flag, and a `rationale` for each).

In [ ]:
# Commit Precommit 4: the failure-mode-to-augmentation mapping.
# One entry per taxonomy category. Each entry has:
#   - augmentations: a list of augmentation kinds meant to counter that mode
#                    (empty for the two unaddressable categories)
#   - addressable:   whether augmentation is the right tool for this mode
#   - rationale:     why this mapping (or why it's left unaddressed)
#
# NB02 reads this mapping and builds the targeted pipeline from the
# `augmentations` lists. The two addressable=False entries are carried on
# the record per Consideration 2 — named, not dropped.
precommit["failure_mode_mapping"] = {
    "background_attended": {
        "augmentations": ["background_randomization"],
        "addressable": True,
        # Cover at minimum: why background variation counters this mode, and
        # what "background_randomization" means concretely (NB02 implements it).
        "rationale": "PLACEHOLDER — REWRITE BEFORE SUBMISSION.",
    },
    "leaf_shape_attended": {
        "augmentations": ["random_crop", "perspective"],
        "addressable": True,
        # Cover at minimum: why breaking the leaf outline counters this mode,
        # and how leaf shape acts as a host-species (hence disease) shortcut.
        "rationale": "PLACEHOLDER — REWRITE BEFORE SUBMISSION.",
    },
    "lighting_attended": {
        "augmentations": ["brightness", "contrast", "color_jitter"],
        "addressable": True,
        # Cover at minimum: why lighting variation counters this mode, and the
        # PV-studio-lighting vs PD-field-lighting contrast that motivates it.
        "rationale": "PLACEHOLDER — REWRITE BEFORE SUBMISSION.",
    },
    "symptom_attended_but_wrong_class": {
        "augmentations": [],
        "addressable": False,
        # Cover at minimum: why augmentation is the WRONG TOOL here — the model
        # attended to a real symptom but the decision boundary is wrong, which
        # no input-space variation fixes. This is the Consideration 2 exclusion.
        "rationale": "PLACEHOLDER — REWRITE BEFORE SUBMISSION.",
    },
    "other_ambiguous": {
        "augmentations": [],
        "addressable": False,
        # Cover at minimum: why there's NOTHING SPECIFIC TO TARGET — these
        # failures weren't sortable into a named mode, so there's no identified
        # cue to vary. A large share here is a finding about the taxonomy.
        "rationale": "PLACEHOLDER — REWRITE BEFORE SUBMISSION.",
    },
}

# Print back what just got committed: which categories are addressed, which
# are deliberately not, and the augmentations attached to each addressed one.
fmm = precommit["failure_mode_mapping"]
print("Failure-mode-to-augmentation mapping:")
for category, entry in fmm.items():
    if entry["addressable"]:
        augs = ", ".join(entry["augmentations"])
        print(f"  [addressed]     {category}: {augs}")
    else:
        print(f"  [not addressed] {category}: (none, by design)")

n_addressed = sum(1 for e in fmm.values() if e["addressable"])
n_excluded = sum(1 for e in fmm.values() if not e["addressable"])
print(f"\n{n_addressed} categories addressed, {n_excluded} deliberately not.")

### 6) Precommit 5: statistical comparison (specific test, not just "compare")

"Outperforms" needs a specific test, not a comparison of two accuracy numbers. The whole point of R2-Q3 is the head-to-head, and a head-to-head that comes down to "targeted got 71% and kitchen-sink got 69%, so targeted wins" isn't a result — it's a coin flip dressed up as a finding. You commit the test here, before you see any numbers, so the verdict rule is fixed in advance.

Three reasonable choices:

- **Bootstrap confidence interval on the gap-reduction difference** — resample the evaluation set many times, recompute each condition's PV→PD gap on each resample, and build a confidence interval around the *difference* in gap reduction between targeted and kitchen-sink. If the interval excludes zero, targeted reduced the gap more than kitchen-sink did. This is the default below: it's robust to small samples, it makes no distributional assumptions, and it reuses the bootstrap machinery R2-Q1 already established.
- **McNemar's test** — a paired test on the per-image agreement between two classifiers (where they agree, where one is right and the other wrong). Appropriate when you're comparing two models on the *same* test images and care about the pattern of disagreements. Gives a p-value rather than an interval.
- **Paired comparison via repeated runs** — train each condition several times with different seeds and compare the distributions of gap reduction. Most expensive (many training runs), but it accounts for training-time randomness, which the other two don't.

The default is the bootstrap CI. It answers the question the page actually asks — "did targeted reduce the gap *more than kitchen-sink did*" — directly, as an interval on the difference, and it degrades gracefully when the per-failure-mode counts get small in NB03.

Commit the verdict rule too, not just the test. A test gives you a number; the verdict rule says what counts as "targeted outperforms." For the bootstrap CI default, the natural rule is "the 95% confidence interval on (targeted gap reduction − kitchen-sink gap reduction) excludes zero." Write down the rule now so that in Week 4 you're reading the result against a fixed bar, not deciding where the bar goes after you see where the number landed.

And hold the page's framing in mind when you set the rule: a *small* margin is informative, not disappointing. If the interval is narrow and sits just above zero, that's the "kitchen-sink incidentally picked up the same signal" result the Prediction anticipates — a real finding, not a failure. Your verdict rule decides statistical significance; the interpretation of a small-but-real margin is a separate matter you'll handle in NB03.

In [ ]:
# Commit Precommit 5: the statistical comparison.
# Default is a bootstrap confidence interval on the difference in gap
# reduction between targeted and kitchen-sink. The verdict rule is committed
# alongside the test — what counts as "targeted outperforms" is fixed now,
# not after the numbers are in.
#
# If you prefer a different test, set "test" to "mcnemar" or "paired_runs"
# and adjust "verdict_rule" to match (e.g. a p-value threshold for McNemar's).
precommit["statistical_comparison"] = {
    "test": "bootstrap_ci",   # one of: "bootstrap_ci", "mcnemar", "paired_runs"
    "comparison_target": "targeted_minus_kitchen_sink_gap_reduction",
    "bootstrap_params": {
        "n_resamples": 2000,   # bootstrap resamples
        "ci_level": 0.95,      # confidence level
    },
    "verdict_rule": "95% CI on (targeted - kitchen_sink) gap reduction excludes zero",
    "reasoning": (
        # Replace this placeholder with your own reasoning before submission.
        # Cover at minimum: which test you chose and why it fits a
        # targeted-vs-kitchen-sink comparison (not just targeted-vs-no-aug);
        # the exact verdict rule and why that bar; and an acknowledgment that
        # a small-but-real margin is the informative result the page predicts,
        # not a null finding.
        "PLACEHOLDER — REWRITE BEFORE SUBMISSION."
    ),
}

# Print back what just got committed.
sc = precommit["statistical_comparison"]
print(f"Statistical test:   {sc['test']}")
print(f"Comparison target:  {sc['comparison_target']}")
if sc["test"] == "bootstrap_ci":
    bp = sc["bootstrap_params"]
    print(f"  Bootstrap: n_resamples={bp['n_resamples']}, ci_level={bp['ci_level']}")
print(f"Verdict rule:       {sc['verdict_rule']}")

### 7) Closeout: write `precommit.json`; submit EQ#1

Week 1 is done once this notebook has written your five decisions to `precommit.json`. The cell below writes the file and reads it straight back to confirm it round-trips — if the write succeeded, the file on disk matches the dict you built section by section.

One thing to do before you submit: the five `reasoning` fields are still placeholders. The decisions themselves have sensible defaults you may have kept as-is, but the reasoning is yours to write — it's the part of EQ#1 that shows you understand *why* each decision is what it is. The summary cell at the end lists exactly which reasoning fields still need rewriting. The notebook won't stop you from submitting with placeholders left in, but a `precommit.json` full of "PLACEHOLDER" is not a finished Week-1 deliverable.

When you open NB01 next week, the first thing it does is read this file back. Anything you change between now and then changes your methodology, not just your result.

In [ ]:
# Write precommit.json to OUTPUT_DIR, then read it back to confirm round-trip.
import json

precommit_path = OUTPUT_DIR / "precommit.json"
with open(precommit_path, "w") as f:
    json.dump(precommit, f, indent=2)

print(f"Wrote {precommit_path}")

# Sanity-load: read the file back and confirm it matches the in-memory dict.
with open(precommit_path, "r") as f:
    precommit_loaded = json.load(f)

assert precommit_loaded == precommit, \
    "Round-trip mismatch — precommit.json does not match the in-memory precommit dict."
print("Sanity-load passed: precommit.json round-trips to the in-memory dict.")